# Sionna 0.19 – Main Ray Tracing Simulation
**Kernel:** `sionna019` · Python 3.10 · Sionna 0.19.2 · TensorFlow 2.15

Migrated from Untitled(1).ipynb (Sionna 2.0) to Sionna 0.19.2 API.
Uses `scene.compute_paths()` and `scene.coverage_map()` (not PathSolver/RadioMapSolver).
GPS / UTM transforms, DEM lookup, TX/RX CSV loading all preserved from original.

## CELL 0 · Environment Setup & Imports

In [ ]:
import os, sys, json, csv, time, warnings, glob, re
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from scipy import stats
from scipy.stats import spearmanr
from scipy.constants import speed_of_light as C
from pyproj import Transformer
from datetime import datetime

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except ImportError:
    pass

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'TF GPU  : {gpus[0].name}')
else:
    print('TF GPU  : NOT detected – running on CPU')
    print('          To fix: check CUDA/cuDNN paths or run:')
    print('          conda install -c conda-forge cudatoolkit cudnn')
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)

import sionna
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver

_HAS_OFDM = False
try:
    from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies
    _HAS_OFDM = True; print('OFDM    : OK  (sionna.channel)')
except (ImportError, AttributeError):
    try:
        from sionna.channel.ofdm import cir_to_ofdm_channel, subcarrier_frequencies
        _HAS_OFDM = True; print('OFDM    : OK  (sionna.channel.ofdm)')
    except: print('OFDM    : NOT found – power fallback will be used')

# ── Mitsuba – try variants in order of preference ─────────────────────────────
# Available in this build: scalar_rgb, scalar_spectral, cuda_ad_rgb, llvm_ad_rgb
# Polarized variants (mono_polarized) not compiled in this build.
_HAS_MI = False
_MI_VARIANT_PREFERENCE = [
    'cuda_ad_rgb',          # GPU + autodiff (best for diff-RT)
    'llvm_ad_rgb',          # CPU + autodiff (fallback)
    'scalar_rgb',           # CPU scalar (ray-cast only, no gradients)
]
try:
    import mitsuba as mi
    for _var in _MI_VARIANT_PREFERENCE:
        try:
            mi.set_variant(_var)
            _HAS_MI = True
            print(f'Mitsuba : {mi.variant()}')
            break
        except Exception:
            continue
    if not _HAS_MI:
        print(f'Mitsuba : imported but no usable variant found')
        print(f'          Available: {", ".join(_MI_VARIANT_PREFERENCE)}')
except ImportError:
    print('Mitsuba : NOT installed')

_HAS_RIO = False
try:
    import rasterio as rio; _HAS_RIO = True; print('rasterio: OK')
except ImportError:
    print('rasterio: NOT available')

_HAS_OSM = False
try:
    import osmnx as ox
    from shapely.geometry import box, Polygon, MultiPolygon
    from shapely.ops import unary_union
    _HAS_OSM = True; print('osmnx   : OK')
except ImportError:
    print('osmnx   : NOT available – pip install osmnx shapely')

print(f'Python  : {sys.version.split()[0]}')
print(f'TF      : {tf.__version__}')
print(f'Sionna  : {sionna.__version__}')

def _safe(v):
    if hasattr(v, 'numpy'): return float(v.numpy())
    if hasattr(v, 'item'):  return float(v.item())
    return float(v)

def _to_numpy(t):
    if isinstance(t, tuple): return t[0].numpy() + 1j * t[1].numpy()
    if hasattr(t, 'numpy'): return t.numpy()
    return np.array(t)

def _cm_to_numpy(cm_obj):
    for attr in ('path_gain', 'rss', 'as_tensor'):
        if not hasattr(cm_obj, attr): continue
        val = getattr(cm_obj, attr)
        arr = val() if callable(val) else val
        if hasattr(arr, 'numpy'): return arr.numpy()
        try: return np.array(arr)
        except: pass
    raise AttributeError('Cannot extract path_gain from CoverageMap.')

## CELL 0b · Install Missing Dependencies

Run once if packages are missing.

In [ ]:
# Uncomment and run once, then restart kernel
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
#     'osmnx', 'shapely', 'pyvista', 'open3d', 'rasterio', 'pyproj',
#     'ipyleaflet', 'ipyvolume', 'tqdm'])
# print('Done – restart kernel.')

## CELL 0c · City Bounding Box

**Edit this cell to switch cities.** Keep the OSM download area ≤ 4 km × 4 km for speed.

The full Nottingham scene points you provided span ~17 km × 9 km (≈93k buildings).
A cropped city-centre tile is used for OSM download; full bounds are kept for Sionna scene.

In [ ]:
# ── City selector ─────────────────────────────────────────────────────────────
CITY_NAME = 'Nottingham'

# ── Full scene bbox (from sionna_web scene points) ────────────────────────────
# Point 0: (-1.447449, 52.918218)
# Point 1: (-1.447449, 53.001149)
# Point 2: (-1.206779, 53.001149)
# Point 3: (-1.206779, 52.918218)
SCENE_WEST   = -1.447449
SCENE_EAST   = -1.206779
SCENE_SOUTH  =  52.918218
SCENE_NORTH  =  53.001149

# Active scene bounds (full area)
WEST, EAST, SOUTH, NORTH = SCENE_WEST, SCENE_EAST, SCENE_SOUTH, SCENE_NORTH

center_lon = (WEST  + EAST)  / 2
center_lat = (SOUTH + NORTH) / 2

# ── Coordinate system for UK (zone 30N) ──────────────────────────────────────
UTM_EPSG = 32630   # WGS84 / UTM zone 30N  (UK)
BNG_EPSG = 27700   # British National Grid  (for UK DEM TIFFs)

# ── Project paths ─────────────────────────────────────────────────────────────
BASE_DIR  = os.path.expanduser(f'~/Documents/FYP2026/{CITY_NAME.lower()}')
OUT_DIR   = os.path.join(BASE_DIR, 'results_sionna019')
SCENE_DIR = BASE_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# DEM TIF – Nottingham terrain
DEM_TIFF    = '/home/georgeskai/Documents/Region/nottingham3602/uk_terrain_nottingham_aoi.tif'

SCENE_XML   = os.path.join(SCENE_DIR, 'scene', 'scene.xml')
TX_CSV      = os.path.join(SCENE_DIR, 'transmitter_positions.csv')
RX_CSV      = os.path.join(SCENE_DIR, 'receiver_locations.csv')
PARAMS_JSON = os.path.join(SCENE_DIR, 'scene_parameters.json')

# ── RF parameters ─────────────────────────────────────────────────────────────
FREQUENCY_HZ  = 3.6e9
BANDWIDTH_HZ  = 20e6
TX_POWER_DBM  = 43.0
TX_GAIN_DBI   = 0.0
RX_GAIN_DBI   = 0.0
LNA_GAIN_DB   = 0.0
NOISE_FLOOR   = -120.0
EIRP_DBM      = TX_POWER_DBM + TX_GAIN_DBI

NUM_SUBCARRIERS    = 76
SUBCARRIER_SPACING = 30e3
if _HAS_OFDM:
    FREQUENCIES = subcarrier_frequencies(NUM_SUBCARRIERS, SUBCARRIER_SPACING)

MAX_DEPTH      = 5
NUM_SAMPLES_CM = 5_000_000
NUM_SAMPLES_PS = 2_000_000
GRID_SIZE_M    = 5.0

_tx_w    = 10**((TX_POWER_DBM - 30) / 10)
_noise_w = 10**((NOISE_FLOOR  - 30) / 10)
SNR_SCALE = _tx_w / _noise_w

# ── Scene area size (sanity check) ────────────────────────────────────────────
_gps_to_utm_tmp = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
_sw = _gps_to_utm_tmp.transform(WEST,  SOUTH)
_ne = _gps_to_utm_tmp.transform(EAST,  NORTH)
_w_km = (_ne[0] - _sw[0]) / 1000
_h_km = (_ne[1] - _sw[1]) / 1000

print('=' * 60)
print(f'CITY          : {CITY_NAME}')
print(f'Scene bbox    : lon [{WEST:.6f}, {EAST:.6f}]')
print(f'                lat [{SOUTH:.6f}, {NORTH:.6f}]')
print(f'Area          : {_w_km:.2f} km x {_h_km:.2f} km')
print(f'Center        : ({center_lon:.6f}, {center_lat:.6f})')
print(f'UTM EPSG      : {UTM_EPSG}')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'DEM           : {DEM_TIFF}')
print(f'Output dir    : {OUT_DIR}')
print('=' * 60)

if _w_km > 10 or _h_km > 10:
    print(f'NOTE: Large area ({_w_km:.1f}x{_h_km:.1f} km). OSM download may take ~1 hour.')


## CELL 1 · Coordinate Utilities + DEM Elevation

- `gps_to_local(lon, lat)` → UTM → subtract scene origin → local XY
- `local_to_gps(x, y)` → reverse
- `get_dem_elevation(local_x, local_y)` → rasterio bilinear lookup
- `ray_cast_ground_z(x, y)` → Mitsuba ray intersect for terrain height

In [ ]:
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', f'EPSG:{BNG_EPSG}', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'UTM center : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())

def get_dem_elevation(local_x, local_y):
    if dem_data is None: return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    px, py = (utm_to_bng.transform(utm_x, utm_y) if _is_bng_dem
              else utm_to_gps.transform(utm_x, utm_y))
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

def ray_cast_ground_z(x, y, max_height=2000.0):
    if _HAS_MI:
        try:
            ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                           mi.Vector3f(0.0, 0.0, -1.0))
            si = scene.mi_scene.ray_intersect(ray)
            if si.is_valid():
                z_val = si.p.z
                return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
        except Exception: pass
    return get_dem_elevation(x, y)

print('Coordinate utilities ready.')
print(f'  gps_to_local({center_lon:.4f}, {center_lat:.4f}) → {gps_to_local(center_lon, center_lat)[:2]}')

## CELL 2 · OSM Map Download

Downloads buildings from OpenStreetMap for the **cropped OSM bbox** defined in CELL 0c.

**Why cropped?** The full Nottingham scene bbox (~17 km × 9 km) contains ~93,000 buildings.
The city-centre tile (~3 km × 3 km) contains ~2,000–5,000 buildings and downloads in seconds.

**Output:** `osm_buildings.geojson` + a Mitsuba 3–compatible `scene.xml` for Sionna.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ── Paths ─────────────────────────────────────────────────────────────────────
OSM_GEOJSON = os.path.join(OUT_DIR, 'osm_buildings.geojson')
OSM_XML     = os.path.join(SCENE_DIR, 'scene', 'scene.xml')
os.makedirs(os.path.dirname(OSM_XML), exist_ok=True)

print(f'OSM download area : lon [{WEST:.5f}, {EAST:.5f}]  lat [{SOUTH:.5f}, {NORTH:.5f}]')
print(f'                  : {_w_km:.2f} km × {_h_km:.2f} km')

if not _HAS_OSM:
    print('\nosmnx not available. Install with:\n  pip install osmnx shapely')
else:
    import geopandas as gpd

    # ── Download or load from cache ───────────────────────────────────────────
    if os.path.exists(OSM_GEOJSON):
        print(f'\nLoading cached buildings from {OSM_GEOJSON} ...')
        gdf = gpd.read_file(OSM_GEOJSON)
        print(f'  Loaded {len(gdf)} buildings from cache.')
    else:
        print('\nDownloading buildings from OpenStreetMap ...')
        t0 = time.time()

        # osmnx 2.0.x API change:
        #   OLD (1.x): ox.features_from_bbox(north=N, south=S, east=E, west=W, tags=tags)
        #   NEW (2.0): ox.features_from_bbox(bbox=(west, south, east, north), tags=tags)
        try:
            ox.settings.log_level = 30   # WARNING only — suppress INFO spam (osmnx 2.0)
        except AttributeError:
            try: ox.settings.log_console = False   # fallback for osmnx 1.x
            except: pass
        ox.settings.timeout = 180

        gdf = ox.features_from_bbox(
            bbox=(WEST, SOUTH, EAST, NORTH),   # osmnx 2.0: (left, bottom, right, top)
            tags={'building': True}
        )
        print(f'  Downloaded {len(gdf)} features in {time.time()-t0:.1f}s')

        # Filter: keep only polygon footprints ≥ 20 m²
        gdf = gdf[gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
        gdf = gdf.to_crs('EPSG:4326')
        gdf_utm = gdf.to_crs(f'EPSG:{UTM_EPSG}')
        gdf     = gdf[gdf_utm.geometry.area >= 20.0].reset_index(drop=True)
        print(f'  After polygon + area filter: {len(gdf)} buildings')

        # Cache to GeoJSON (geometry only, avoids serialisation issues)
        gdf[['geometry']].to_file(OSM_GEOJSON, driver='GeoJSON')
        print(f'  Saved to {OSM_GEOJSON}')

    # ── Building height extraction ─────────────────────────────────────────────
    # OSM tags checked in order: 'height', 'building:height', 'building:levels'
    DEFAULT_HEIGHT_M = 10.0   # fallback when no tag present
    LEVEL_HEIGHT_M   = 3.0    # metres per floor

    def _parse_height(row):
        for col in ('height', 'building:height'):
            if col in row and pd.notna(row[col]):
                try: return float(str(row[col]).split()[0])
                except: pass
        if 'building:levels' in row and pd.notna(row.get('building:levels')):
            try: return float(row['building:levels']) * LEVEL_HEIGHT_M
            except: pass
        return DEFAULT_HEIGHT_M

    heights = gdf.apply(_parse_height, axis=1).values
    n_tagged = int((heights != DEFAULT_HEIGHT_M).sum())
    print(f'\n  Building heights: min={heights.min():.1f}  mean={heights.mean():.1f}  '
          f'max={heights.max():.1f} m  ({n_tagged}/{len(gdf)} tagged)')

    # ── Plot ───────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Map — colour by height
    gdf.plot(ax=axes[0], column=heights, cmap='YlOrRd',
             legend=True, legend_kwds={'label': 'Height (m)'},
             edgecolor='k', linewidth=0.2, alpha=0.8)
    axes[0].add_patch(Rectangle((WEST, SOUTH), EAST-WEST, NORTH-SOUTH,
                                 fill=False, edgecolor='blue', lw=2, label='OSM bbox'))
    axes[0].set_title(f'{CITY_NAME} – OSM Buildings ({len(gdf):,})')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
    axes[0].legend()

    # Height histogram
    axes[1].hist(heights, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(DEFAULT_HEIGHT_M, color='red', ls='--',
                    label=f'Default fallback ({DEFAULT_HEIGHT_M} m)')
    axes[1].set_xlabel('Building Height (m)'); axes[1].set_ylabel('Count')
    axes[1].set_title('Building Height Distribution'); axes[1].legend()

    plt.suptitle(f'{CITY_NAME} OSM Download  |  {len(gdf):,} buildings', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'osm_buildings.png'), dpi=150)
    plt.show()
    print(f'Map saved → {os.path.join(OUT_DIR, "osm_buildings.png")}')

## CELL 3 · Generate Mitsuba 3 Scene XML from OSM Buildings

Converts OSM footprints + heights into a Sionna 0.19-compatible Mitsuba 3 scene XML.
Each building is extruded as a box mesh. Ground plane is added automatically.

> Skip this cell if you already have a `scene.xml` from sionna_web.

In [ ]:
# OSM -> Mitsuba 3 XML with ITU-R materials + DEM elevation (Sionna 0.19)
# Building bases are raised to actual terrain height from DEM_TIFF.
# Ground is a DEM-sampled terrain mesh (64x64 grid).

import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom
import numpy as np
import os, time

_HAS_DEM_RASTERIO = False
try:
    import rasterio
    _HAS_DEM_RASTERIO = True
except ImportError:
    pass

if not _HAS_OSM or 'gdf' not in dir():
    print('Run CELL 2 first to download OSM buildings.')
else:
    print(f'Generating Mitsuba 3 scene XML from {len(gdf)} buildings ...')
    t0 = time.time()

    SCENE_OUT_DIR = os.path.join(BASE_DIR, 'scene')
    os.makedirs(SCENE_OUT_DIR, exist_ok=True)
    SCENE_XML_OUT = os.path.join(SCENE_OUT_DIR, 'scene.xml')

    # ── Load DEM ──────────────────────────────────────────────────────────
    _dem_ds = _dem_data = _dem_tf = _wgs_to_dem = None

    if _HAS_DEM_RASTERIO and os.path.exists(DEM_TIFF):
        try:
            from pyproj import Transformer as _Tr
            _dem_ds   = rasterio.open(DEM_TIFF)
            _dem_data = _dem_ds.read(1).astype(np.float32)
            _dem_tf   = _dem_ds.transform
            _epsg_str = str(_dem_ds.crs.to_epsg()) if _dem_ds.crs else ''
            _wgs_to_dem = _Tr.from_crs('EPSG:4326',
                              f'EPSG:{_epsg_str}' if _epsg_str else _dem_ds.crs,
                              always_xy=True)
            print(f'  DEM: {_dem_data.shape} px, CRS EPSG:{_epsg_str}')
        except Exception as _e:
            print(f'  DEM load failed: {_e}')
            _dem_ds = None
    else:
        print(f'  DEM not found at {DEM_TIFF} -- buildings sit at Z=0')

    def _dem_z(lon, lat):
        if _dem_ds is None:
            return 0.0
        try:
            dx, dy = _wgs_to_dem.transform(lon, lat)
            tf = _dem_tf
            col = (dx - tf.c) / tf.a
            row = (dy - tf.f) / tf.e
            r, c = int(round(row)), int(round(col))
            nr, nc = _dem_data.shape
            r = max(0, min(nr - 1, r))
            c = max(0, min(nc - 1, c))
            v = float(_dem_data[r, c])
            return v if (np.isfinite(v) and v > -9000) else 0.0
        except Exception:
            return 0.0

    origin_elev = _dem_z(center_lon, center_lat)

    # ── Project OSM to UTM ────────────────────────────────────────────────
    gdf_utm = gdf.to_crs(f'EPSG:{UTM_EPSG}')
    ox_utm, oy_utm = utm_center_x, utm_center_y

    # ── XML root ──────────────────────────────────────────────────────────
    root = ET.Element('scene', version='3.0.0')
    for name, val in [
        ('scenegen_min_lon',    str(WEST)),
        ('scenegen_max_lon',    str(EAST)),
        ('scenegen_min_lat',    str(SOUTH)),
        ('scenegen_max_lat',    str(NORTH)),
        ('scenegen_origin_lon', str(center_lon)),
        ('scenegen_origin_lat', str(center_lat)),
    ]:
        ET.SubElement(root, 'default', name=name, value=val)

    integ = ET.SubElement(root, 'integrator', type='path')
    ET.SubElement(integ, 'integer', name='max_depth', value='8')

    # ── ITU-R materials ───────────────────────────────────────────────────
    ITU_MATERIALS = {
        'itu_concrete'          : {'relative_permittivity': '5.24',  'conductivity': '0.130'},
        'itu_brick'             : {'relative_permittivity': '3.91',  'conductivity': '0.024'},
        'itu_glass'             : {'relative_permittivity': '6.27',  'conductivity': '0.012'},
        'itu_wood'              : {'relative_permittivity': '1.99',  'conductivity': '0.005'},
        'itu_medium_dry_ground' : {'relative_permittivity': '15.0',  'conductivity': '0.035'},
        'itu_asphalt'           : {'relative_permittivity': '3.00',  'conductivity': '0.010'},
        'itu_vegetation'        : {'relative_permittivity': '1.30',  'conductivity': '0.001'},
    }
    for mat_id, props in ITU_MATERIALS.items():
        bsdf = ET.SubElement(root, 'bsdf', type='diffuse', id=mat_id)
        ET.SubElement(bsdf, 'rgb', name='reflectance', value='0.5 0.5 0.5')
        for k, v in props.items():
            ET.SubElement(bsdf, 'float', name=k, value=v)

    # ── Terrain mesh (DEM 64x64 grid) ─────────────────────────────────────
    DEM_GRID_N = 64
    lons_g = np.linspace(WEST,  EAST,  DEM_GRID_N)
    lats_g = np.linspace(SOUTH, NORTH, DEM_GRID_N)

    terrain_shape = ET.SubElement(root, 'shape', type='obj', id='terrain')
    ET.SubElement(terrain_shape, 'ref', id='itu_medium_dry_ground')
    t_lines = []
    for lat in lats_g:
        for lon in lons_g:
            ux, uy = gps_to_utm.transform(lon, lat)
            lx, ly = ux - ox_utm, uy - oy_utm
            z = _dem_z(lon, lat) - origin_elev
            t_lines.append(f'v {lx:.3f} {ly:.3f} {z:.3f}')
    for r in range(DEM_GRID_N - 1):
        for c in range(DEM_GRID_N - 1):
            i00 = r * DEM_GRID_N + c + 1
            i10 = i00 + 1
            i01 = i00 + DEM_GRID_N
            i11 = i01 + 1
            t_lines.append(f'f {i00} {i10} {i11} {i01}')
    ET.SubElement(terrain_shape, 'string', name='mesh_data', value='\n'.join(t_lines))

    # ── OSM building heights ───────────────────────────────────────────────
    heights = []
    for _, row_g in gdf.iterrows():
        try:
            if row_g.get('height') and str(row_g['height']).replace('.','',1).isdigit():
                heights.append(float(row_g['height']))
            elif row_g.get('building:levels'):
                heights.append(float(row_g['building:levels']) * 3.0)
            else:
                heights.append(8.0)
        except (TypeError, ValueError):
            heights.append(8.0)

    def _bld_mat(row):
        mat = str(row.get('building:material', '')).lower()
        tag = str(row.get('building', '')).lower()
        if 'glass' in mat:                       return 'itu_glass'
        if 'wood' in mat or 'timber' in mat:     return 'itu_wood'
        if 'brick' in mat:                       return 'itu_brick'
        if tag in ('greenhouse', 'glasshouse'):  return 'itu_glass'
        return 'itu_concrete'

    def _make_building(idx, geom_utm, height, base_z, mat_id):
        try:
            hull = geom_utm.convex_hull
            ring = list(hull.exterior.coords) if hull.geom_type == 'Polygon' else None
        except Exception:
            return None
        if ring is None:
            return None
        pts = [(x - ox_utm, y - oy_utm) for x, y in ring[:-1]]
        n = len(pts)
        if n < 3:
            return None
        z0, z1 = base_z, base_z + height
        verts = [(x, y, z0) for x, y in pts] + [(x, y, z1) for x, y in pts]
        faces = []
        for i in range(n):
            j = (i + 1) % n
            faces += [(i, j, j+n), (i, j+n, i+n)]
        for i in range(1, n-1):
            faces.append((n, n+i, n+i+1))
        for i in range(1, n-1):
            faces.append((0, i+1, i))
        s = ET.Element('shape', type='obj', id=f'building_{idx:05d}')
        ET.SubElement(s, 'ref', id=mat_id)
        lines = [f'v {x:.3f} {y:.3f} {z:.3f}' for x, y, z in verts]
        lines += ['f ' + ' '.join(str(i+1) for i in f) for f in faces]
        ET.SubElement(s, 'string', name='mesh_data', value='\n'.join(lines))
        return s

    gdf_wgs_list = list(gdf.iterrows())
    gdf_utm_list = list(gdf_utm.iterrows())
    skipped = 0
    for idx in range(len(gdf_utm_list)):
        _, row_utm = gdf_utm_list[idx]
        _, row_wgs = gdf_wgs_list[idx]
        geom = row_utm.geometry
        if geom is None or geom.is_empty:
            skipped += 1; continue
        try:
            ctr = row_wgs.geometry.centroid
            base_z = _dem_z(ctr.x, ctr.y) - origin_elev
        except Exception:
            base_z = 0.0
        h  = heights[idx]
        mat = _bld_mat(dict(row_wgs))
        s  = _make_building(idx, geom, h, base_z, mat)
        if s is None:
            skipped += 1; continue
        root.append(s)

    n_built = sum(1 for s in root.iter('shape') if s.get('id','').startswith('building_'))
    print(f'  {n_built} building shapes  ({skipped} skipped)')

    xml_str = minidom.parseString(
        ET.tostring(root, encoding='unicode')
    ).toprettyxml(indent='  ', encoding=None)

    with open(SCENE_XML_OUT, 'w', encoding='utf-8') as f:
        f.write(xml_str)

    SCENE_XML = SCENE_XML_OUT
    XML_OK    = True
    if _dem_ds:
        _dem_ds.close()

    print(f'  Done in {time.time()-t0:.1f}s')
    print(f'  Scene XML -> {SCENE_XML_OUT}')
    print('Materials: itu_concrete/brick/glass/wood | terrain: itu_medium_dry_ground')
    print('Ready for CELL 4: load_scene(SCENE_XML)')


## CELL 4 · Load 3-D Scene & Configure Antennas

**Sionna 0.19 API:** `load_scene(path)` — no `merge_shapes` argument.

In [ ]:
XML_OK = os.path.exists(SCENE_XML)
if not XML_OK:
    raise RuntimeError(
        f'scene.xml not found at {SCENE_XML}\n'
        'Either:\n'
        '  A) Run CELL 3 to generate from OSM (requires Blender for full mesh)\n'
        '  B) Generate via sionna_web Steps 1-3\n'
        '  C) Use Blender + BlenderOSM plugin')

print(f'Loading scene from {SCENE_XML} ...')
scene = load_scene(SCENE_XML)   # Sionna 0.19: NO merge_shapes argument
scene.frequency = FREQUENCY_HZ

scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')

print(f'Scene loaded  : {len(scene.objects)} objects,  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'Materials     : {list(scene.radio_materials.keys())}')

## CELL 5 · Assign ITU-R P.2040-2 Material Properties

In [ ]:
_ITU_DB = {
    'concrete'          : (5.24,  0.130, 0.40, 0.20),
    'brick'             : (3.91,  0.024, 0.30, 0.20),
    'wood'              : (1.99,  0.005, 0.25, 0.30),
    'glass'             : (6.27,  0.012, 0.08, 0.10),
    'metal'             : (1.00,  1e7,   0.05, 0.10),
    'asphalt'           : (3.00,  0.010, 0.35, 0.20),
    'vegetation'        : (1.30,  0.001, 0.75, 0.05),
    'water'             : (81.0,  0.500, 0.02, 0.05),
    'wet_ground'        : (30.0,  0.150, 0.20, 0.20),
    'medium_dry_ground' : (15.0,  0.035, 0.18, 0.20),
    'very_dry_ground'   : (3.00,  0.001, 0.12, 0.20),
    'marble'            : (7.07,  0.020, 0.08, 0.10),
    'plasterboard'      : (2.73,  0.010, 0.12, 0.20),
}
_DEFAULT_MAT = (4.0, 0.08, 0.30, 0.15)

def _match_itu(mat_name):
    n = mat_name.lower().replace('itu_', '').replace('mat-', '').replace('mat_', '')
    for key in _ITU_DB:
        if key in n: return key
    for key in _ITU_DB:
        if any(part in n for part in key.split('_')): return key
    return None

print('ASSIGNING ITU-R MATERIAL PROPERTIES')
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    eps_r, sigma, S, xpd = _ITU_DB.get(key, _DEFAULT_MAT)
    try: mat.relative_permittivity = eps_r
    except: pass
    try: mat.conductivity = sigma
    except: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, S); break
            except: pass
    print(f'  {mat_name:<32}  matched={key or "DEFAULT":<20}  eps={eps_r:.2f}  σ={sigma:.4g}')
print('Done.')

## CELL 6 · Load Transmitter (GPS → local XY, ray-cast Z)

In [ ]:
for nm in list(scene.transmitters.keys()): scene.remove(nm)

if os.path.exists(TX_CSV):
    df_tx    = pd.read_csv(TX_CSV)
    row      = df_tx.iloc[0]
    tx_name  = str(row.get('name', 'tx_0'))
    tx_lon   = float(row['lon'])
    tx_lat   = float(row['lat'])
    tx_agl   = float(row.get('height', 25.0))
    tx_power = float(row.get('power_dbm', EIRP_DBM))
else:
    # Default TX at scene centre
    tx_name = 'tx0'; tx_lon = center_lon; tx_lat = center_lat
    tx_agl  = 25.0;  tx_power = EIRP_DBM
    print('TX CSV not found – using scene centre')

local_x, local_y, _ = gps_to_local(tx_lon, tx_lat)
ground_z = ray_cast_ground_z(local_x, local_y)
abs_z    = ground_z + tx_agl

tx = Transmitter(name=tx_name,
                 position=(float(local_x), float(local_y), float(abs_z)),
                 power_dbm=float(tx_power))
scene.add(tx)
print(f'✓ TX "{tx_name}"  GPS=({tx_lon:.5f}, {tx_lat:.5f})  '
      f'XY=({local_x:.1f}, {local_y:.1f})  Z={abs_z:.1f} m  EIRP={tx_power:.1f} dBm')

## CELL 7 · Load Receivers (GPS → local XY, ray-cast Z)

In [ ]:
for nm in list(scene.receivers.keys()): scene.remove(nm)
receivers = []

if os.path.exists(RX_CSV):
    df_rx = pd.read_csv(RX_CSV)
    print(f'Loaded {len(df_rx)} receivers from CSV ...')
    t0 = time.time()
    for i, row in df_rx.iterrows():
        lon = float(row['lon']); lat = float(row['lat'])
        agl = float(row.get('height', 1.5))
        x, y, _ = gps_to_local(lon, lat)
        z = ray_cast_ground_z(x, y) + agl
        nm = str(row.get('name', f'RX_{i+1:04d}'))
        rx = Receiver(name=nm, position=(float(x), float(y), float(z)))
        scene.add(rx); receivers.append(rx)
    print(f'Done in {time.time()-t0:.2f}s')
else:
    rx = Receiver(name='rx0', position=(100.0, 0.0, abs_z - tx_agl + 1.5))
    scene.add(rx); receivers.append(rx)
    print('RX CSV not found – single default receiver placed')

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')
for rx in receivers[:5]:
    x, y, z = _safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name:<15} XY=({x:8.1f},{y:8.1f})  Z={z:.2f}  GPS=({lon:.5f},{lat:.5f})')

## CELL 8 · Coverage Map (Sionna 0.19 API)

In [ ]:
try:
    _bbox = scene.mi_scene.bbox()
    cx = (float(_bbox.min[0]) + float(_bbox.max[0])) / 2
    cy = (float(_bbox.min[1]) + float(_bbox.max[1])) / 2
    gx_min, gx_max = float(_bbox.min[0]), float(_bbox.max[0])
    gy_min, gy_max = float(_bbox.min[1]), float(_bbox.max[1])
except Exception:
    cx = cy = 0.0; gx_min = gy_min = -500.0; gx_max = gy_max = 500.0

print('Computing coverage map WITH scattering ...')
cm_scatter = scene.coverage_map(
    cm_cell_size=GRID_SIZE_M, max_depth=MAX_DEPTH, num_samples=NUM_SAMPLES_CM,
    los=True, specular_reflection=True, diffuse_reflection=True,
    refraction=True, diffraction=False)

print('Computing coverage map WITHOUT scattering ...')
cm_no_scatter = scene.coverage_map(
    cm_cell_size=GRID_SIZE_M, max_depth=MAX_DEPTH, num_samples=NUM_SAMPLES_CM,
    los=True, specular_reflection=True, diffuse_reflection=False,
    refraction=False, diffraction=False)

_tx_dbm       = float(tx.power_dbm) if hasattr(tx, 'power_dbm') else TX_POWER_DBM
cm_s_np       = _cm_to_numpy(cm_scatter)
cm_ns_np      = _cm_to_numpy(cm_no_scatter)
rssi_scatter    = 10*np.log10(cm_s_np[0]  + 1e-30) + _tx_dbm
rssi_no_scatter = 10*np.log10(cm_ns_np[0] + 1e-30) + _tx_dbm

tx_x = _safe(tx.position[0]); tx_y = _safe(tx.position[1])
vmin, vmax = -120, -40

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, data, title in [
    (axes[0], rssi_scatter,    f'With scattering @ {FREQUENCY_HZ/1e9:.2f} GHz'),
    (axes[1], rssi_no_scatter, f'Without scattering @ {FREQUENCY_HZ/1e9:.2f} GHz'),
]:
    im = ax.imshow(data, origin='lower', extent=[gx_min, gx_max, gy_min, gy_max],
                   cmap='jet', aspect='auto', vmin=vmin, vmax=vmax)
    ax.scatter(tx_x, tx_y, marker='*', s=300, c='gold', edgecolors='black', label='TX')
    sample_step = max(1, len(receivers)//200)
    ax.scatter([_safe(r.position[0]) for r in receivers[::sample_step]],
               [_safe(r.position[1]) for r in receivers[::sample_step]],
               s=5, c='cyan', alpha=0.5, label='RX')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_title(title); ax.legend()
    plt.colorbar(im, ax=ax, label='RSSI (dBm)')
plt.suptitle(f'{CITY_NAME} Coverage Map', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'coverage_map_comparison.png'), dpi=150)
plt.show()

## CELL 9 · Path Computation (Sionna 0.19 API)

In [ ]:
print(f'Computing paths: depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')
paths = scene.compute_paths(
    max_depth=MAX_DEPTH, num_samples=NUM_SAMPLES_PS,
    los=True, specular_reflection=True, diffuse_reflection=True,
    refraction=True, diffraction=False)
print('Done.')

a_np = _to_numpy(paths.a)
if a_np.ndim == 6: a_np = a_np[0]

power  = np.sum(np.abs(a_np)**2, axis=tuple(range(1, a_np.ndim)))
pg_db  = 10 * np.log10(power + 1e-30)
rssi_rx = pg_db + _tx_dbm
n_rx    = len(pg_db)

print(f'Path gain ({n_rx} RX): mean={pg_db.mean():.1f}  min={pg_db.min():.1f}  max={pg_db.max():.1f} dB')

if _HAS_OFDM:
    try:
        h_freq = cir_to_ofdm_channel(FREQUENCIES, *paths.cir(), normalize=False)
        print(f'OFDM channel shape : {_to_numpy(h_freq).shape}')
    except Exception as e:
        print(f'OFDM CIR failed: {e}')

## CELL 10 · Per-Receiver Results CSV (GPS coordinates restored)

In [ ]:
H, W = rssi_scatter.shape
XX, YY = np.meshgrid(np.linspace(gx_min, gx_max, W),
                     np.linspace(gy_min, gy_max, H))
tree = KDTree(np.column_stack([XX.ravel(), YY.ravel()]))
rx_coords = np.array([(_safe(rx.position[0]), _safe(rx.position[1])) for rx in receivers])
_, idx    = tree.query(rx_coords)
pg_cm_db  = rssi_scatter.ravel()[idx]

records = []
for i, rx in enumerate(receivers[:n_rx]):
    x, y, z = _safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    records.append({
        'receiver'    : rx.name,
        'lon'         : round(lon, 6),
        'lat'         : round(lat, 6),
        'x_m'         : round(x, 2),
        'y_m'         : round(y, 2),
        'z_m'         : round(z, 3),
        'rssi_cm_dbm' : round(float(pg_cm_db[i]), 2) if i < len(pg_cm_db) else float('nan'),
        'rssi_ps_dbm' : round(float(rssi_rx[i]),  2) if i < len(rssi_rx)  else float('nan'),
        'pg_db'       : round(float(pg_db[i]),     2) if i < len(pg_db)   else float('nan'),
    })

df_out = pd.DataFrame(records)
out_csv = os.path.join(OUT_DIR, 'receiver_results.csv')
df_out.to_csv(out_csv, index=False)
print(f'Saved {len(df_out)} receivers to {out_csv}')
print(df_out.head(10).to_string(index=False))

## CELL 11 · Path Loss vs Distance

In [ ]:
tx_pos2d = np.array([_safe(tx.position[0]), _safe(tx.position[1])])
dist_m   = np.linalg.norm(df_out[['x_m','y_m']].values - tx_pos2d, axis=1)
df_out['dist_m'] = dist_m
df_out['path_loss'] = -df_out['pg_db']

_lam  = C / FREQUENCY_HZ
d_ref = np.linspace(max(dist_m.min(), 10), dist_m.max(), 300)
fspl  = 20*np.log10(4*np.pi*d_ref / _lam)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(dist_m, df_out['path_loss'], s=5, alpha=0.4, c='steelblue', label='Simulated')
ax.plot(d_ref, fspl, 'r--', lw=2, label='Free-space PL')
ax.set_xlabel('Distance TX→RX (m)'); ax.set_ylabel('Path Loss (dB)')
ax.set_title(f'{CITY_NAME} – Path Loss @ {FREQUENCY_HZ/1e9:.2f} GHz')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'path_loss_vs_distance.png'), dpi=150)
plt.show()
print('All results saved to:', OUT_DIR)